# iPhone 사진 YOLO 분석
같은 위치의 `iphone` 폴더에 있는 모든 사진을 `best_v8.pt` 모델로 YOLO 탐지하고 결과를 출력합니다.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

BASE_DIR = Path.cwd()
MODEL_PATH = BASE_DIR / "best_v8.pt"
IMAGE_DIR = BASE_DIR / "iphone"
CONFIDENCE = 0.25

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"모델 파일이 없습니다: {MODEL_PATH.resolve()}")
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {IMAGE_DIR.resolve()}")

device = 0 if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO(str(MODEL_PATH))

In [ ]:
image_paths = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"분석할 이미지: {len(image_paths)}장")
for p in image_paths:
    print(" -", p.name)

In [ ]:
for image_path in image_paths:
    results = model.predict(
        source=str(image_path),
        conf=CONFIDENCE,
        device=device,
        verbose=False,
    )
    result = results[0]

    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 8))
    plt.imshow(annotated)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

    boxes = result.boxes
    if boxes is not None and len(boxes) > 0:
        print(f"[{image_path.name}] 탐지 {len(boxes)}개:")
        for cls_id, conf in zip(boxes.cls.tolist(), boxes.conf.tolist()):
            print(f"  - {result.names[int(cls_id)]}: {conf:.2f}")
    else:
        print(f"[{image_path.name}] 탐지된 객체 없음")
    print()